<a href="https://colab.research.google.com/github/MJMortensonWarwick/AnalyticsEngineering2026/blob/main/4_03_pandas_cleaning_and_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 4.03 Cleaning and processing data with pandas
A working reference for the pandas functions you will use most when turning messy raw data into something you can trust. It uses the card-spend data from the bank case: card transactions, customer snapshots and exchange rates.

Run the cells in order. Each section does one job, shows the function, and ends with a line saying when you would use it. The last section collects everything into a lookup table.

**Sections**
1. Loading and first look
2. Selecting and filtering
3. Data types
4. Missing values
5. Duplicates
6. Text cleaning
7. Recoding and creating columns
8. Dates and times
9. Sorting and ranking
10. Grouping and aggregating
11. Window and running calculations
12. Reshaping
13. Combining tables
14. Validation and outliers
15. Chaining steps into a pipeline
16. Saving results
17. Quick reference

## 0. Setup

In [1]:
import glob, os, urllib.request, zipfile
import numpy as np
import pandas as pd

print("pandas", pd.__version__)

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/testgitsession/card-spend-practice.zip"
DATA_DIR = "/content/card-spend"

if not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/card-spend.zip")
    zipfile.ZipFile("/content/card-spend.zip").extractall(DATA_DIR)

def find(name):
    hits = glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True)
    if not hits:
        raise FileNotFoundError(f"Missing {name}. Ask for help.")
    return hits[0]

pandas 2.2.3


## 1. Loading and first look
Before changing anything, find out what you have. Most cleaning mistakes come from skipping this step.

`pd.read_csv` has options that save cleaning later: `dtype` (force a type), `usecols` (read only some columns), `parse_dates`, `na_values` (extra strings that mean missing), `nrows` (read a sample).

In [2]:
tx = pd.read_csv(find("card_transactions.csv"), dtype={"ref_txn_id": "string"})
cu = pd.read_csv(find("customer_master.csv"))
fx = pd.read_csv(find("fx_rates.csv"))
print(tx.shape, cu.shape, fx.shape)   # (rows, columns)

(28829, 9) (5600, 7) (198, 3)


In [3]:
# Read only what you need, and only a few rows to test
pd.read_csv(find("card_transactions.csv"), usecols=["txn_id", "amount", "currency"], nrows=5)

,txn_id,amount,currency
0,T000001,132.66,GBP
1,T000002,41.65,GBP
2,T000003,30.22,GBP
3,T000004,4.42,GBP
4,T000005,1.86,GBP


In [4]:
tx.head()          # first rows; tx.tail() for the last rows

,txn_id,ref_txn_id,card_id,merchant,amount,currency,status,txn_timestamp,file_date
0,T000001,<NA>,C1090,AMAZON MKTPLACE,132.66,GBP,PENDING,2026-01-01 00:03:06,2026-01-01
1,T000002,<NA>,C1037,WAITROSE 5418,41.65,GBP,PENDING,2026-01-01 00:25:35,2026-01-01
2,T000003,<NA>,C1058,SAINSBURYS S/MKT 7861,30.22,GBP,PENDING,01/01/2026 00:29,2026-01-01
3,T000004,<NA>,C1341,GREGGS 7178,4.42,GBP,PENDING,2026-01-01 00:36:25,2026-01-01
4,T000005,<NA>,C1241,COSTA COFFEE 1172,1.86,GBP,PENDING,2026-01-01 02:34:39,2026-01-01


In [5]:
tx.sample(5, random_state=1)   # random rows: better than head() when the file is sorted

,txn_id,ref_txn_id,card_id,merchant,amount,currency,status,txn_timestamp,file_date
9914,T004889,<NA>,C1378,SHELL 9924,35.66,GBP,POSTED,2026-01-30 08:04:29,2026-02-01
36,T000038,<NA>,C1296,PRET A MANGER 8631,6.02,GBP,PENDING,2026-01-01 08:29:48,2026-01-01
6503,T003409,<NA>,C1117,COSTA COFFEE 4026,2.11,GBP,POSTED,2026-01-21 13:35:39,2026-01-22
11355,T006115,<NA>,C1075,TRAINLINE,24.51,GBP,PENDING,2026-02-06 16:48:49,2026-02-06
18962,T010017,<NA>,C1131,amazon mktplace,37.12,GBP,PENDING,2026-03-02 14:13:53,2026-03-02


In [6]:
tx.info()          # column names, types, non-null counts, memory

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28829 entries, 0 to 28828
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   txn_id         28829 non-null  object 
 1   ref_txn_id     295 non-null    string 
 2   card_id        28829 non-null  object 
 3   merchant       28829 non-null  object 
 4   amount         28829 non-null  float64
 5   currency       28829 non-null  object 
 6   status         28829 non-null  object 
 7   txn_timestamp  28829 non-null  object 
 8   file_date      28829 non-null  object 
dtypes: float64(1), object(7), string(1)
memory usage: 2.0+ MB


In [7]:
tx.dtypes          # types only

,0
txn_id,object
ref_txn_id,string[python]
card_id,object
merchant,object
amount,float64
currency,object
status,object
txn_timestamp,object
file_date,object


In [8]:
tx.describe()      # summary of numeric columns

,amount
count,28829.000000
mean,31.980011
std,43.750476
min,-399.930000
25%,10.130000
50%,20.580000
75%,39.520000
max,973.120000


In [9]:
tx.select_dtypes(exclude="number").describe()   # summary of text columns (or describe(include="all") for everything)

,txn_id,ref_txn_id,card_id,merchant,currency,status,txn_timestamp,file_date
count,28829,295,28829,28829,28829,28829,28829,28829
unique,15126,281,408,503,10,3,15094,87
top,T012276,T013765,C1331,AMAZON MKTPLACE,GBP,POSTED,16/02/2026 17:12,2026-02-13
freq,3,2,106,2351,25885,15059,8,1170


In [10]:
tx.nunique()       # distinct values per column

,0
txn_id,15126
ref_txn_id,281
card_id,408
merchant,503
amount,6577
currency,10
status,3
txn_timestamp,15094
file_date,87


In [11]:
tx.isna().sum()    # missing values per column

,0
txn_id,0
ref_txn_id,28534
card_id,0
merchant,0
amount,0
currency,0
status,0
txn_timestamp,0
file_date,0


In [12]:
tx["status"].value_counts()            # count of each value

,count
status,
POSTED,15059
PENDING,13475
REVERSED,295


In [13]:
tx["currency"].value_counts(dropna=False)   # note the dirty codes: "gbp", "GBP "

,count
currency,
GBP,25885
EUR,1306
USD,918
CHF,406
GBP,156
gbp,136
eur,9
USD,8
EUR,3


In [14]:
tx["status"].value_counts(normalize=True).round(3)   # proportions instead of counts

,proportion
status,
POSTED,0.522
PENDING,0.467
REVERSED,0.010


In [15]:
tx.columns.tolist(), tx.index

(['txn_id',
  'ref_txn_id',
  'card_id',
  'merchant',
  'amount',
  'currency',
  'status',
  'txn_timestamp',
  'file_date'],
 RangeIndex(start=0, stop=28829, step=1))

**Use when:** the first five minutes with any new table. Look for wrong types, unexpected categories, missing values and odd ranges before you plan anything.

## 2. Selecting and filtering
Select columns with `[]`, rows by label with `.loc`, rows by position with `.iloc`. Filter rows with a boolean mask (a column of True/False).

In [16]:
tx["amount"]                       # one column -> a Series
tx[["txn_id", "amount"]].head(3)   # several columns -> a DataFrame

,txn_id,amount
0,T000001,132.66
1,T000002,41.65
2,T000003,30.22


In [17]:
tx.loc[0:2, ["txn_id", "merchant"]]    # .loc: labels (end is INCLUDED)

,txn_id,merchant
0,T000001,AMAZON MKTPLACE
1,T000002,WAITROSE 5418
2,T000003,SAINSBURYS S/MKT 7861


In [18]:
tx.iloc[0:3, 0:2]                      # .iloc: positions (end is EXCLUDED)

,txn_id,ref_txn_id
0,T000001,<NA>
1,T000002,<NA>
2,T000003,<NA>


In [19]:
tx[tx["amount"] > 500].head(3)         # boolean mask

,txn_id,ref_txn_id,card_id,merchant,amount,currency,status,txn_timestamp,file_date
2779,T001659,<NA>,C1294,BOOKING.COM,973.12,EUR,PENDING,2026-01-11 07:04:09,2026-01-11
5067,T002933,<NA>,C1361,BOOKING.COM,671.87,EUR,PENDING,2026-01-18 15:52:35,2026-01-18
5822,T002933,<NA>,C1361,BOOKING.COM,671.87,EUR,POSTED,2026-01-18 15:52:35,2026-01-20


In [20]:
# Combine conditions with & (and), | (or), ~ (not). Each condition needs brackets.
tx[(tx["status"] == "POSTED") & (tx["amount"] > 500)].shape

(9, 9)

In [21]:
tx[tx["currency"].isin(["EUR", "USD"])].shape       # value in a list

(2224, 9)

In [22]:
tx[tx["amount"].between(100, 200)].shape            # inclusive range

(974, 9)

In [23]:
tx[tx["merchant"].str.contains("TESCO", na=False)].shape   # text search

(2967, 9)

In [24]:
tx.query("status == 'POSTED' and amount > 500").shape      # same idea, readable string form

(9, 9)

In [25]:
tx.loc[tx["amount"] == 0, ["txn_id", "amount", "status"]].head()   # filter rows AND choose columns

,txn_id,amount,status
874,T000364,0.0,POSTED
1509,T000729,0.0,POSTED
3269,T001457,0.0,POSTED
3361,T001696,0.0,POSTED
3595,T001616,0.0,POSTED


In [26]:
tx.drop(columns=["ref_txn_id"]).columns.tolist()    # drop columns (returns a new table)

['txn_id',
 'card_id',
 'merchant',
 'amount',
 'currency',
 'status',
 'txn_timestamp',
 'file_date']

In [27]:
tx.rename(columns={"txn_timestamp": "ts_raw"}).columns.tolist()   # rename columns

['txn_id',
 'ref_txn_id',
 'card_id',
 'merchant',
 'amount',
 'currency',
 'status',
 'ts_raw',
 'file_date']

In [28]:
tx[["amount", "txn_id"]].columns.tolist()          # reorder columns by listing them in the order you want

['amount', 'txn_id']

**Copies.** Filtering and most methods return a new table; the original is unchanged unless you assign the result back. When you want to change a filtered table, take an explicit copy first so pandas does not warn about changing a view:

In [29]:
posted = tx[tx["status"] == "POSTED"].copy()
posted.shape

(15059, 9)

**Use when:** any time you need a subset. Prefer `.loc` with a mask when you also want to assign to those rows (section 7).

## 3. Data types
Wrong types cause silent errors: numbers stored as text sort wrongly (`"100"` before `"20"`), dates stored as text cannot be subtracted, and text that should be a category wastes memory.

In [30]:
# A small messy table to practise on
messy = pd.DataFrame({
    "id": ["1", "2", "3", "4"],
    "amount": ["10.50", "£20.00", "n/a", "1,200"],
    "flag": ["yes", "no", "yes", "no"],
})
messy.dtypes

,0
id,object
amount,object
flag,object


In [31]:
messy["id"] = messy["id"].astype(int)             # astype: convert (fails if impossible)
messy.dtypes

,0
id,int64
amount,object
flag,object


In [32]:
# to_numeric with errors="coerce": anything that cannot convert becomes NaN instead of crashing
pd.to_numeric(messy["amount"], errors="coerce")

,amount
0,10.5
1,NaN
2,NaN
3,NaN


In [33]:
# Clean the text first, then convert
cleaned = (messy["amount"]
           .str.replace("£", "", regex=False)
           .str.replace(",", "", regex=False))
pd.to_numeric(cleaned, errors="coerce")

,amount
0,10.5
1,20.0
2,NaN
3,1200.0


In [34]:
messy["flag"].map({"yes": True, "no": False})    # text to boolean

,flag
0,True
1,False
2,True
3,False


In [35]:
# category: for columns with few distinct values (saves memory, allows an order)
tx["status"].astype("category").dtype, tx["status"].memory_usage(deep=True), tx["status"].astype("category").memory_usage(deep=True)

(CategoricalDtype(categories=['PENDING', 'POSTED', 'REVERSED'], ordered=False, categories_dtype=object),
 1599792,
 29237)

In [36]:
# An ordered category sorts in your order, not alphabetical
order = pd.CategoricalDtype(["PENDING", "POSTED", "REVERSED"], ordered=True)
tx["status"].astype(order).sort_values().head(3)

,status
0,PENDING
15663,PENDING
15664,PENDING


In [37]:
# Nullable integer: keeps whole numbers even when values are missing (capital I)
pd.Series([1, 2, None]).astype("Int64")

,0
0,1
1,2
2,<NA>


In [38]:
# Downcast numbers to save memory
tx["amount"].astype("float32").memory_usage(), tx["amount"].memory_usage()

(115448, 230764)

In [39]:
# Convert many columns at once
tx.astype({"amount": "float32", "status": "category"}).dtypes.head(6)

,0
txn_id,object
ref_txn_id,string[python]
card_id,object
merchant,object
amount,float32
currency,object


In [40]:
# Let pandas infer better types (useful after loading)
tx.convert_dtypes().dtypes.head(6)

,0
txn_id,string[python]
ref_txn_id,string[python]
card_id,string[python]
merchant,string[python]
amount,Float64
currency,string[python]


**Use when:** straight after loading. Check `dtypes`, convert, and use `errors="coerce"` followed by a count of the new NaNs so you can see what failed to convert.

## 4. Missing values
Missing data is represented as `NaN` (numbers), `NaT` (dates) or `<NA>` (nullable types). The key decision is what a missing value *means* here: unknown, not applicable, or an error. The code follows the decision, not the other way round.

The card data has few real gaps, so this section uses a small table.

In [41]:
gaps = pd.DataFrame({
    "card_id": ["C1", "C1", "C1", "C2", "C2", "C3"],
    "day": pd.date_range("2026-03-01", periods=6),
    "balance": [100.0, np.nan, 120.0, np.nan, np.nan, 50.0],
    "note": ["ok", None, "ok", "", "ok", None],
})
gaps

,card_id,day,balance,note
0,C1,2026-03-01,100.0,ok
1,C1,2026-03-02,NaN,None
2,C1,2026-03-03,120.0,ok
3,C2,2026-03-04,NaN,
4,C2,2026-03-05,NaN,ok
5,C3,2026-03-06,50.0,None


In [42]:
gaps.isna()                 # True where missing

,card_id,day,balance,note
0,False,False,False,False
1,False,False,True,True
2,False,False,False,False
3,False,False,True,False
4,False,False,True,False
5,False,False,False,True


In [43]:
gaps.isna().sum()           # count per column

,0
card_id,0
day,0
balance,3
note,2


In [44]:
gaps.notna().sum()          # count of present values

,0
card_id,6
day,6
balance,3
note,4


In [45]:
gaps.isna().mean().round(2) # share missing per column

,0
card_id,0.00
day,0.00
balance,0.50
note,0.33


In [46]:
gaps[gaps["balance"].isna()]   # the rows with a missing balance

,card_id,day,balance,note
1,C1,2026-03-02,NaN,None
3,C2,2026-03-04,NaN,
4,C2,2026-03-05,NaN,ok


In [47]:
gaps.dropna()               # drop rows with ANY missing value

,card_id,day,balance,note
0,C1,2026-03-01,100.0,ok
2,C1,2026-03-03,120.0,ok


In [48]:
gaps.dropna(subset=["balance"])   # drop rows missing in chosen columns only

,card_id,day,balance,note
0,C1,2026-03-01,100.0,ok
2,C1,2026-03-03,120.0,ok
5,C3,2026-03-06,50.0,None


In [49]:
gaps.dropna(thresh=3)       # keep rows with at least 3 non-missing values

,card_id,day,balance,note
0,C1,2026-03-01,100.0,ok
2,C1,2026-03-03,120.0,ok
3,C2,2026-03-04,NaN,
4,C2,2026-03-05,NaN,ok
5,C3,2026-03-06,50.0,None


In [50]:
gaps["balance"].fillna(0)   # fill with a constant

,balance
0,100.0
1,0.0
2,120.0
3,0.0
4,0.0
5,50.0


In [51]:
gaps["balance"].fillna(gaps["balance"].mean())   # fill with a statistic

,balance
0,100.0
1,90.0
2,120.0
3,90.0
4,90.0
5,50.0


In [52]:
gaps["balance"].ffill()     # carry the last known value forward (the FX weekend rule)

,balance
0,100.0
1,100.0
2,120.0
3,120.0
4,120.0
5,50.0


In [53]:
gaps["balance"].bfill()     # carry the next known value backward

,balance
0,100.0
1,120.0
2,120.0
3,50.0
4,50.0
5,50.0


In [54]:
# Forward fill WITHIN each group, so one card's balance never leaks into another's
gaps.assign(balance_filled=gaps.groupby("card_id")["balance"].ffill())

,card_id,day,balance,note,balance_filled
0,C1,2026-03-01,100.0,ok,100.0
1,C1,2026-03-02,NaN,None,100.0
2,C1,2026-03-03,120.0,ok,120.0
3,C2,2026-03-04,NaN,,NaN
4,C2,2026-03-05,NaN,ok,NaN
5,C3,2026-03-06,50.0,None,50.0


In [55]:
gaps["balance"].interpolate()   # straight-line fill between known points

,balance
0,100.000000
1,110.000000
2,120.000000
3,96.666667
4,73.333333
5,50.000000


In [56]:
# Empty strings are NOT missing to pandas. Convert them first.
gaps["note"].replace("", np.nan).isna().sum(), gaps["note"].isna().sum()

(np.int64(3), np.int64(2))

In [57]:
# Fill from another column (first non-missing wins)
gaps["note"].replace("", np.nan).fillna(gaps["card_id"])

,note
0,ok
1,C1
2,ok
3,C2
4,ok
5,C3


In [58]:
# Keep a record of what you filled
gaps.assign(balance_was_missing=gaps["balance"].isna(), balance=gaps["balance"].ffill())

,card_id,day,balance,note,balance_was_missing
0,C1,2026-03-01,100.0,ok,False
1,C1,2026-03-02,100.0,None,True
2,C1,2026-03-03,120.0,ok,False
3,C2,2026-03-04,120.0,,True
4,C2,2026-03-05,120.0,ok,True
5,C3,2026-03-06,50.0,None,False


**Use when:** after loading and again after joins (a left join creates missing values). Record what you filled so a user can tell real data from filled data.

## 5. Duplicates
Decide first what makes two rows "the same". An exact duplicate repeats every column. A *key* duplicate repeats the id but may differ elsewhere (for example a later status).

In [59]:
tx.duplicated().sum()            # rows identical to an earlier row

np.int64(1269)

In [60]:
tx[tx.duplicated(keep=False)].sort_values("txn_id").head(6)   # see BOTH copies

,txn_id,ref_txn_id,card_id,merchant,amount,currency,status,txn_timestamp,file_date
9076,R00055,T003524,C1398,GREGGS 5015,-3.74,GBP,REVERSED,2026-01-29 07:54:00,2026-01-29
8762,R00055,T003524,C1398,GREGGS 5015,-3.74,GBP,REVERSED,2026-01-29 07:54:00,2026-01-29
8763,R00069,T003931,C1068,AMAZON MKTPLACE,-41.95,GBP,REVERSED,2026-01-29 15:33:00,2026-01-29
9077,R00069,T003931,C1068,AMAZON MKTPLACE,-41.95,GBP,REVERSED,2026-01-29 15:33:00,2026-01-29
8764,R00070,T003937,C1341,GREGGS 7178,-1.43,GBP,REVERSED,2026-01-29 10:14:00,2026-01-29
9078,R00070,T003937,C1341,GREGGS 7178,-1.43,GBP,REVERSED,2026-01-29 10:14:00,2026-01-29


In [61]:
tx.drop_duplicates().shape       # remove exact duplicates (keeps the first)

(27560, 9)

In [62]:
tx["txn_id"].duplicated().sum()  # repeated ids (different statuses over time)

np.int64(13703)

In [63]:
tx.drop_duplicates(subset=["txn_id"]).shape   # same id counts as a duplicate: keeps the FIRST row per id, which is not necessarily the latest

(15126, 9)

In [64]:
# Keep the LATEST row per id: sort, then keep the last one
latest = (tx.drop_duplicates()
            .sort_values(["txn_id", "file_date"])
            .drop_duplicates(subset="txn_id", keep="last"))
latest.shape

(15126, 9)

In [65]:
# Same result with groupby (useful when the rule is more than "last")
tx.drop_duplicates().sort_values(["txn_id", "file_date"]).groupby("txn_id").tail(1).shape

(15126, 9)

In [66]:
# Check that an id is now unique: this should print True
latest["txn_id"].is_unique

True

**Use when:** the source can resend data. Always count before and after, and check whether the rows you drop are really identical.

## 6. Text cleaning
All text methods live under `.str` and work on a whole column at once. They return missing for missing input, so you rarely need a loop.

In [67]:
tx["currency"].unique()                # spot the problems first

array(['GBP', 'gbp', 'EUR', 'GBP ', 'CHF', 'USD', 'eur', 'usd', 'USD ',
       'EUR '], dtype=object)

In [68]:
tx["currency"].str.strip().str.upper().value_counts()   # strip spaces, upper-case

,count
currency,
GBP,26177
EUR,1318
USD,928
CHF,406


In [69]:
s = pd.Series(["  Premium ", "premium", "PREMIUM", "Standard  "])
s.str.strip().str.title()     # title case: "Premium"

,0
0,Premium
1,Premium
2,Premium
3,Standard


In [70]:
s.str.lower(), s.str.len()   # lower-case and length

(0      premium 
 1       premium
 2       premium
 3    standard  
 dtype: object,
 0    10
 1     7
 2     7
 3    10
 dtype: int64)

In [71]:
tx["merchant"].value_counts().head(8)   # note the store numbers

,count
merchant,
AMAZON MKTPLACE,2351
TFL TRAVEL CH,2175
NETFLIX.COM,1068
SPOTIFY,1007
TRAINLINE,776
TESCO STORES 4848,722
TESCO STORES 5181,706
TESCO STORES 3549,688


In [72]:
# Collapse repeated spaces with a regular expression
pd.Series(["TESCO   STORES  4848"]).str.replace(r"\s+", " ", regex=True)

,0
0,TESCO STORES 4848


In [73]:
# Remove trailing store numbers: "TESCO STORES 4848" -> "TESCO STORES"
tx["merchant"].str.replace(r"\s*\d+$", "", regex=True).str.strip().nunique(), tx["merchant"].nunique()

(331, 503)

In [74]:
# Extract part of a string with a capture group
tx["merchant"].str.extract(r"(\d+)$").head()

,0
0,NaN
1,5418
2,7861
3,7178
4,1172


In [75]:
tx["merchant"].str.startswith("TESCO").sum(), tx["merchant"].str.contains("S/MKT", regex=False).sum()

(np.int64(2967), np.int64(2401))

In [76]:
# Split a string into pieces
tx["txn_timestamp"].str.split(" ", expand=True).head(3)

,0,1
0,2026-01-01,00:03:06
1,2026-01-01,00:25:35
2,01/01/2026,00:29


In [77]:
# Slice characters: first 7 characters of an ISO date text
tx["file_date"].str[:7].head(3)

,file_date
0,2026-01
1,2026-01
2,2026-01


In [78]:
pd.Series(["a,b", "c"]).str.split(",").explode()    # one item per row

,0
0,a
0,b
1,c


In [79]:
# Standardise by mapping variants to one value: a small lookup is clearer than a long regex
fix = {"gbp": "GBP", "eur": "EUR", "usd": "USD", "chf": "CHF"}
tx["currency"].str.strip().str.lower().map(fix).value_counts(dropna=False)

,count
currency,
GBP,26177
EUR,1318
USD,928
CHF,406


In [80]:
# Check nothing was lost: unmapped values become NaN, so count them
tx["currency"].str.strip().str.lower().map(fix).isna().sum()

np.int64(0)

**Use when:** labels typed by humans or different systems. Clean to a new column when the original might still be needed (keep `merchant`, add `merchant_clean`).

## 7. Recoding and creating columns
New columns are the core of the silver layer: flags, buckets, converted values. Prefer vectorised operations (whole-column arithmetic, `np.where`, `map`) over `apply` or row loops. They are shorter and far faster.

In [81]:
tx2 = tx.copy()
tx2["is_reversal"] = tx2["status"] == "REVERSED"          # a True/False flag
tx2["is_zero"] = tx2["amount"].eq(0)
tx2[["status", "is_reversal", "amount", "is_zero"]].head(3)

,status,is_reversal,amount,is_zero
0,PENDING,False,132.66,False
1,PENDING,False,41.65,False
2,PENDING,False,30.22,False


In [82]:
# np.where: if / else
tx2["direction"] = np.where(tx2["amount"] < 0, "refund", "purchase")
tx2["direction"].value_counts()

,count
direction,
purchase,28534
refund,295


In [83]:
# np.select: several conditions, first match wins
conditions = [tx2["amount"] < 0, tx2["amount"] == 0, tx2["amount"] < 50, tx2["amount"] < 200]
labels = ["refund", "zero", "small", "medium"]
tx2["size"] = np.select(conditions, labels, default="large")
tx2["size"].value_counts()

,count
size,
small,23541
medium,4595
large,337
refund,295
zero,61


In [84]:
# pd.cut: bins you choose.  pd.qcut: bins with equal numbers of rows.
bands = pd.cut(tx2["amount"], bins=[-1000, 0, 50, 200, 10000], labels=["<=0", "0-50", "50-200", "200+"])
bands.value_counts().sort_index()

,count
amount,
<=0,356
0-50,23543
50-200,4593
200+,337


In [85]:
pd.qcut(tx2["amount"], q=4, labels=["Q1", "Q2", "Q3", "Q4"]).value_counts().sort_index()

,count
amount,
Q1,7214
Q2,7205
Q3,7204
Q4,7206


In [86]:
# map with a dictionary: recode a column
tx2["status_rank"] = tx2["status"].map({"PENDING": 0, "POSTED": 1, "REVERSED": 2})
tx2["status_rank"].value_counts()

,count
status_rank,
1,15059
0,13475
2,295


In [87]:
# replace: swap values, including several at once
tx2["status"].replace({"PENDING": "P", "POSTED": "S"}).value_counts()

,count
status,
S,15059
P,13475
REVERSED,295


In [88]:
# where / mask: keep values where the condition is true, otherwise replace
tx2["amount"].where(tx2["amount"] > 0, 0).min(), tx2["amount"].clip(lower=0).min()

(0.0, 0.0)

In [89]:
# clip: cap values at limits
tx2["amount"].clip(lower=0, upper=500).describe()[["min", "max"]]

,amount
min,0.0
max,500.0


In [90]:
# assign: create columns in a chain without changing the original
tx.assign(amount_x2=tx["amount"] * 2, is_big=lambda d: d["amount"] > 100).head(3)

,txn_id,ref_txn_id,card_id,merchant,amount,currency,status,txn_timestamp,file_date,amount_x2,is_big
0,T000001,<NA>,C1090,AMAZON MKTPLACE,132.66,GBP,PENDING,2026-01-01 00:03:06,2026-01-01,265.32,True
1,T000002,<NA>,C1037,WAITROSE 5418,41.65,GBP,PENDING,2026-01-01 00:25:35,2026-01-01,83.30,False
2,T000003,<NA>,C1058,SAINSBURYS S/MKT 7861,30.22,GBP,PENDING,01/01/2026 00:29,2026-01-01,60.44,False


In [91]:
# apply: a Python function per value. Slow on big tables; use it only when nothing vectorised works.
tx["amount"].head(3).apply(lambda v: round(v * 0.8, 1))

,amount
0,106.1
1,33.3
2,24.2


In [92]:
# Speed comparison: vectorised vs apply on 28,829 rows
import time
t0 = time.time(); _ = tx["amount"] * 0.8; t_vec = time.time() - t0
t0 = time.time(); _ = tx["amount"].apply(lambda v: v * 0.8); t_app = time.time() - t0
print(f"vectorised {t_vec:.4f}s | apply {t_app:.4f}s")

vectorised 0.0020s | apply 0.0337s


In [93]:
# Change values in selected rows only: .loc[mask, column] = value
tx2.loc[tx2["amount"] == 0, "status"] = "ZERO"
tx2["status"].value_counts()

,count
status,
POSTED,14998
PENDING,13475
REVERSED,295
ZERO,61


In [94]:
# Round and format
tx2["amount"].round(1).head(3), tx2["amount"].abs().head(3)

(0    132.7
 1     41.6
 2     30.2
 Name: amount, dtype: float64,
 0    132.66
 1     41.65
 2     30.22
 Name: amount, dtype: float64)

**Use when:** every transformation step. Write the rule in plain words first (for example *a reversal is a row whose status is REVERSED*), then translate it into one of these.

## 8. Dates and times
Dates arrive as text in many formats. Convert once, early, to proper datetimes; after that you can extract parts, subtract, group by month and join on date.

In [95]:
tx["txn_timestamp"].sample(6, random_state=3).tolist()   # two formats are mixed

['2026-01-18 12:39:05',
 '2026-01-05 17:10:46',
 '07/01/2026 09:21',
 '2026-02-05 16:18:44',
 '2026-02-24 01:17:16',
 '2026-03-10 07:55:08']

In [96]:
# Parse each format separately; errors="coerce" gives NaT where a format does not fit
iso = pd.to_datetime(tx["txn_timestamp"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
uk  = pd.to_datetime(tx["txn_timestamp"], format="%d/%m/%Y %H:%M", errors="coerce")
ts = iso.fillna(uk)                       # take the first that worked
print("could not parse:", ts.isna().sum(), "| ISO:", iso.notna().sum(), "| day-first:", uk.notna().sum())

could not parse: 0 | ISO: 25381 | day-first: 3448


In [97]:
# One call alternative: format="mixed" guesses each value. Convenient, but set dayfirst on purpose.
pd.to_datetime(tx["txn_timestamp"].head(4), format="mixed", dayfirst=True)

,txn_timestamp
0,2026-01-01 00:03:06
1,2026-01-01 00:25:35
2,2026-01-01 00:29:00
3,2026-01-01 00:36:25


In [98]:
tx3 = tx.assign(ts=ts)
tx3["ts"].dtype

dtype('<M8[ns]')

In [99]:
# The .dt accessor pulls out parts
d = tx3["ts"].dt
pd.DataFrame({"year": d.year, "month": d.month, "day": d.day, "hour": d.hour,
              "weekday": d.day_name(), "is_weekend": d.dayofweek >= 5}).head(4)

,year,month,day,hour,weekday,is_weekend
0,2026,1,1,0,Thursday,False
1,2026,1,1,0,Thursday,False
2,2026,1,1,0,Thursday,False
3,2026,1,1,0,Thursday,False


In [100]:
tx3["ts"].dt.date.head(2), tx3["ts"].dt.normalize().head(2)   # date only (normalize keeps the datetime type)

(0    2026-01-01
 1    2026-01-01
 Name: ts, dtype: object,
 0   2026-01-01
 1   2026-01-01
 Name: ts, dtype: datetime64[ns])

In [101]:
tx3["month"] = tx3["ts"].dt.to_period("M")             # a month period: groups and sorts properly
tx3["month"].value_counts().sort_index()

,count
month,
2026-01,9982
2026-02,8926
2026-03,9921


In [102]:
tx3["ts"].dt.strftime("%Y-%m").head(3)   # format as text, e.g. for labels

,ts
0,2026-01
1,2026-01
2,2026-01


In [103]:
# Arithmetic: differences give timedeltas
tx3["file_dt"] = pd.to_datetime(tx3["file_date"])
tx3["days_late"] = (tx3["file_dt"] - tx3["ts"].dt.normalize()).dt.days
tx3["days_late"].value_counts().sort_index().head(8)

,count
days_late,
0,13029
1,7692
2,3946
3,3789
4,226
5,90
6,56
7,1


In [104]:
# Filter by date
tx3[(tx3["ts"] >= "2026-03-01") & (tx3["ts"] < "2026-04-01")].shape

(9921, 13)

In [105]:
# Build a full calendar, then see what is missing (this is how FX gaps are found)
fx["rate_date"] = pd.to_datetime(fx["rate_date"])
cal = pd.date_range(fx["rate_date"].min(), "2026-03-31", freq="D")
print(len(cal), "calendar days |", fx["rate_date"].nunique(), "days with rates")

93 calendar days | 66 days with rates


In [106]:
# Pivot to one column per currency, put on the full calendar, then fill the gaps
wide = fx.pivot(index="rate_date", columns="currency", values="rate_to_gbp").reindex(cal)
print("missing days per currency:", wide.isna().sum().to_dict())
wide.ffill().isna().sum().to_dict()

missing days per currency: {'CHF': 27, 'EUR': 27, 'USD': 27}


{'CHF': 0, 'EUR': 0, 'USD': 0}

In [107]:
# Resample: aggregate a time series to a new frequency ("W" weekly, "D" daily; "ME" month-end needs pandas 2.2+)
tx3.set_index("ts")["amount"].resample("W").sum().head(4)

,amount
ts,
2026-01-04,39697.53
2026-01-11,72027.20
2026-01-18,72484.69
2026-01-25,71169.26


In [108]:
# Offsets and shifts
tx3["ts"].head(2) + pd.Timedelta(days=1), tx3["ts"].head(2) + pd.DateOffset(months=1)

(0   2026-01-02 00:03:06
 1   2026-01-02 00:25:35
 Name: ts, dtype: datetime64[ns],
 0   2026-02-01 00:03:06
 1   2026-02-01 00:25:35
 Name: ts, dtype: datetime64[ns])

In [109]:
# Timezones (the data has none; shown for reference)
tx3["ts"].head(2).dt.tz_localize("Europe/London").dt.tz_convert("UTC")

,ts
0,2026-01-01 00:03:06+00:00
1,2026-01-01 00:25:35+00:00


**Use when:** any date column. Decide which date you mean (purchase date, file date, posting date) before you group by month, because the choice changes the totals.

## 9. Sorting and ranking
Sorting is how you decide "latest", "first" and "top". Be explicit about ties.

In [110]:
tx.sort_values("amount", ascending=False).head(3)

,txn_id,ref_txn_id,card_id,merchant,amount,currency,status,txn_timestamp,file_date
2779,T001659,<NA>,C1294,BOOKING.COM,973.12,EUR,PENDING,2026-01-11 07:04:09,2026-01-11
18487,T009301,<NA>,C1105,BOOKING.COM,758.06,USD,POSTED,2026-02-26 06:13:09,2026-02-28
17758,T009301,<NA>,C1105,BOOKING.COM,758.06,USD,PENDING,2026-02-26 06:13:09,2026-02-26


In [111]:
tx.sort_values(["status", "amount"], ascending=[True, False]).head(3)   # several keys

,txn_id,ref_txn_id,card_id,merchant,amount,currency,status,txn_timestamp,file_date
2779,T001659,<NA>,C1294,BOOKING.COM,973.12,EUR,PENDING,2026-01-11 07:04:09,2026-01-11
17758,T009301,<NA>,C1105,BOOKING.COM,758.06,USD,PENDING,2026-02-26 06:13:09,2026-02-26
18126,T009560,<NA>,C1016,BOOKING.COM,691.66,EUR,PENDING,2026-02-27 17:41:09,2026-02-27


In [112]:
tx.nlargest(3, "amount")[["txn_id", "amount"]], tx.nsmallest(3, "amount")[["txn_id", "amount"]]

(        txn_id  amount
 2779   T001659  973.12
 17758  T009301  758.06
 18487  T009301  758.06,
        txn_id  amount
 13999  R00112 -399.93
 11889  R00111 -190.86
 1846   R00011 -175.85)

In [113]:
tx.sort_index().head(2)   # sort by the index; tx.reset_index(drop=True) renumbers rows 0..n-1

,txn_id,ref_txn_id,card_id,merchant,amount,currency,status,txn_timestamp,file_date
0,T000001,<NA>,C1090,AMAZON MKTPLACE,132.66,GBP,PENDING,2026-01-01 00:03:06,2026-01-01
1,T000002,<NA>,C1037,WAITROSE 5418,41.65,GBP,PENDING,2026-01-01 00:25:35,2026-01-01


In [114]:
# rank: position within the column. method="first" gives every row a different rank.
tx["amount"].rank(method="first", ascending=False).head(3)

,amount
0,785.0
1,6664.0
2,10271.0


In [115]:
# Rank within a group: the biggest purchase per card
tx.assign(rank_in_card=tx.groupby("card_id")["amount"].rank(method="first", ascending=False))\
  .query("rank_in_card == 1").shape

(408, 10)

In [116]:
# idxmax / idxmin: the row label of the largest / smallest value
tx.loc[tx.groupby("card_id")["amount"].idxmax(), ["card_id", "txn_id", "amount"]].head(3)

,card_id,txn_id,amount
27962,C1001,T014492,66.36
1623,C1002,T001056,360.95
14423,C1003,T007596,107.63


In [117]:
# Tie-break by a custom order (the same trick the pipeline uses for status)
rank = {"PENDING": 0, "POSTED": 1, "REVERSED": 2}
(tx.assign(r=tx["status"].map(rank))
   .sort_values(["txn_id", "file_date", "r"])
   .groupby("txn_id").tail(1).shape)

(15126, 10)

**Use when:** choosing one row per group. Sort first, then `groupby().tail(1)` or `drop_duplicates(keep="last")`; never rely on the order the file arrived in.

## 10. Grouping and aggregating
`groupby` splits the table into groups, applies a calculation to each, and combines the results. This is how gold tables are made.

In [118]:
tx.groupby("status")["amount"].sum()

,amount
status,
PENDING,440010.50
POSTED,491051.21
REVERSED,-9109.97


In [119]:
tx.groupby("status")["amount"].agg(["count", "sum", "mean", "min", "max"])

,count,sum,mean,min,max
status,,,,,
PENDING,13475,440010.50,32.653840,0.59,973.12
POSTED,15059,491051.21,32.608487,0.00,758.06
REVERSED,295,-9109.97,-30.881254,-399.93,-1.17


In [120]:
# Named aggregation: choose the output column names
tx.groupby("status").agg(n=("txn_id", "count"),
                         total=("amount", "sum"),
                         cards=("card_id", "nunique"))

,n,total,cards
status,,,
PENDING,13475,440010.50,408
POSTED,15059,491051.21,408
REVERSED,295,-9109.97,189


In [121]:
# Several keys
tx.groupby(["currency", "status"]).size().head(6)

currency  status  
CHF       PENDING     190
          POSTED      213
          REVERSED      3
EUR       PENDING     612
          POSTED      681
          REVERSED     13
dtype: int64

In [122]:
# as_index=False keeps the keys as ordinary columns; reset_index does the same afterwards
tx.groupby("status", as_index=False)["amount"].sum()

,status,amount
0,PENDING,440010.50
1,POSTED,491051.21
2,REVERSED,-9109.97


In [123]:
# transform: the group result, but kept at the original row level (for shares and flags)
tx.assign(card_total=tx.groupby("card_id")["amount"].transform("sum"))[["card_id", "amount", "card_total"]].head(3)

,card_id,amount,card_total
0,C1090,132.66,1840.20
1,C1037,41.65,2091.29
2,C1058,30.22,2225.63


In [124]:
# Share of group total
tx.assign(share=tx["amount"] / tx.groupby("card_id")["amount"].transform("sum"))["share"].head(3)

,share
0,0.072090
1,0.019916
2,0.013578


In [125]:
# filter: keep whole groups that pass a test (cards with more than 100 rows)
tx.groupby("card_id").filter(lambda g: len(g) > 100).shape

(724, 9)

In [126]:
# pivot_table: groupby with the second key across the columns
tx.pivot_table(index="status", columns="currency", values="amount", aggfunc="sum", fill_value=0).round(0).iloc[:, :4]

currency,CHF,EUR,EUR,GBP
status,,,,
PENDING,17902.0,53841.0,138.0,327582.0
POSTED,20901.0,60702.0,43.0,363468.0
REVERSED,-260.0,-1422.0,0.0,-6278.0


In [127]:
# crosstab: counts of combinations
pd.crosstab(tx["status"], tx["currency"].str.strip().str.upper())

currency,CHF,EUR,GBP,USD
status,,,,
PENDING,190,618,12233,434
POSTED,213,687,13678,481
REVERSED,3,13,266,13


In [128]:
# value_counts is a groupby in disguise
tx["card_id"].value_counts().head(3)

,count
card_id,
C1331,106
C1071,105
C1295,105


In [129]:
# Several aggregations with a custom function
tx.groupby("status")["amount"].agg(lambda s: s.quantile(0.9))

,amount
status,
PENDING,67.426
POSTED,67.540
REVERSED,-3.848


In [130]:
# Group by a derived key: month of purchase
tx3.groupby(tx3["ts"].dt.to_period("M"))["amount"].agg(["count", "sum"])

,count,sum
ts,,
2026-01,9982,324596.89
2026-02,8926,284047.57
2026-03,9921,313307.28


In [131]:
# first / last / nth within groups
tx.sort_values("file_date").groupby("txn_id").agg(first_status=("status", "first"), last_status=("status", "last")).head(3)

,first_status,last_status
txn_id,,
R00001,REVERSED,REVERSED
R00002,REVERSED,REVERSED
R00003,REVERSED,REVERSED


**Use when:** building summaries. Check the number of rows in equals the number accounted for in the groups (the totals must reconcile).

## 11. Window and running calculations
Window functions look across neighbouring rows: previous value, running total, moving average. Sort first; the order defines "previous".

In [132]:
daily = (tx3.assign(day=tx3["ts"].dt.normalize())
            .loc[lambda d: d["status"] == "POSTED"]
            .groupby("day", as_index=False)["amount"].sum()
            .sort_values("day"))
daily.head(3)

,day,amount
0,2026-01-01,4764.37
1,2026-01-02,5243.03
2,2026-01-03,4801.74


In [133]:
daily.assign(running_total=daily["amount"].cumsum()).head(3)      # cumulative sum

,day,amount,running_total
0,2026-01-01,4764.37,4764.37
1,2026-01-02,5243.03,10007.40
2,2026-01-03,4801.74,14809.14


In [134]:
daily.assign(prev=daily["amount"].shift(1), change=daily["amount"].diff()).head(3)   # previous value and change

,day,amount,prev,change
0,2026-01-01,4764.37,NaN,NaN
1,2026-01-02,5243.03,4764.37,478.66
2,2026-01-03,4801.74,5243.03,-441.29


In [135]:
daily.assign(pct=daily["amount"].pct_change().round(3)).head(3)   # percent change

,day,amount,pct
0,2026-01-01,4764.37,NaN
1,2026-01-02,5243.03,0.100
2,2026-01-03,4801.74,-0.084


In [136]:
daily.assign(avg7=daily["amount"].rolling(7).mean().round(1)).iloc[5:9]    # 7-day moving average (NaN until 7 rows)

,day,amount,avg7
5,2026-01-06,5598.33,NaN
6,2026-01-07,5217.25,5341.7
7,2026-01-08,5573.55,5457.3
8,2026-01-09,4502.92,5351.6


In [137]:
daily.assign(avg7=daily["amount"].rolling(7, min_periods=1).mean().round(1)).head(3)

,day,amount,avg7
0,2026-01-01,4764.37,4764.4
1,2026-01-02,5243.03,5003.7
2,2026-01-03,4801.74,4936.4


In [138]:
daily.assign(expanding_max=daily["amount"].expanding().max()).head(3)

,day,amount,expanding_max
0,2026-01-01,4764.37,4764.37
1,2026-01-02,5243.03,5243.03
2,2026-01-03,4801.74,5243.03


In [139]:
# Within groups: previous transaction amount for the same card
t = tx3.sort_values(["card_id", "ts"])
t.assign(prev_amount=t.groupby("card_id")["amount"].shift(1))[["card_id", "ts", "amount", "prev_amount"]].head(4)

,card_id,ts,amount,prev_amount
136,C1001,2026-01-02 00:54:04,21.61,NaN
1073,C1001,2026-01-02 00:54:04,21.61,21.61
645,C1001,2026-01-04 08:09:25,9.80,21.61
1426,C1001,2026-01-04 08:09:25,9.80,9.80


In [140]:
# Time between consecutive transactions on a card
t.assign(gap=t.groupby("card_id")["ts"].diff())["gap"].describe()

,gap
count,28421
mean,1 days 04:51:05.430456352
std,2 days 01:28:33.208094955
min,0 days 00:00:00
25%,0 days 00:00:00
50%,0 days 01:23:12
75%,1 days 16:41:52
max,25 days 09:37:09


In [141]:
# Running count within a group
t.assign(n_so_far=t.groupby("card_id").cumcount() + 1)[["card_id", "n_so_far"]].head(4)

,card_id,n_so_far
136,C1001,1
1073,C1001,2
645,C1001,3
1426,C1001,4


**Use when:** time-based features, finding gaps and detecting changes (for example a segment that changed between snapshots).

## 12. Reshaping
Wide tables have one column per category (good for reading); long tables have one row per observation (good for joining and grouping). Tools turn one into the other.

In [142]:
wide.head(3)   # wide: one column per currency (from section 8)

currency,CHF,EUR,USD
2025-12-29,0.9112,0.8628,0.7877
2025-12-30,0.9062,0.8703,0.7854
2025-12-31,0.9108,0.8690,0.7852


In [143]:
# melt: wide to long
long = wide.reset_index().melt(id_vars="index", var_name="currency", value_name="rate").rename(columns={"index": "rate_date"})
long.head(4)

,rate_date,currency,rate
0,2025-12-29,CHF,0.9112
1,2025-12-30,CHF,0.9062
2,2025-12-31,CHF,0.9108
3,2026-01-01,CHF,NaN


In [144]:
# pivot: long to wide. It fails if there are duplicates, which is a useful check.
long.pivot(index="rate_date", columns="currency", values="rate").head(3)

currency,CHF,EUR,USD
rate_date,,,
2025-12-29,0.9112,0.8628,0.7877
2025-12-30,0.9062,0.8703,0.7854
2025-12-31,0.9108,0.8690,0.7852


In [145]:
# pivot_table: like pivot but aggregates duplicates
long.pivot_table(index="rate_date", columns="currency", values="rate", aggfunc="mean").head(2)

currency,CHF,EUR,USD
rate_date,,,
2025-12-29,0.9112,0.8628,0.7877
2025-12-30,0.9062,0.8703,0.7854


In [146]:
# stack / unstack: move the innermost column level into the rows and back
summary = tx.groupby(["status", "currency"])["amount"].sum()
summary.unstack().round(0).iloc[:, :4]

currency,CHF,EUR,EUR,GBP
status,,,,
PENDING,17902.0,53841.0,138.0,327582.0
POSTED,20901.0,60702.0,43.0,363468.0
REVERSED,-260.0,-1422.0,NaN,-6278.0


In [147]:
tx.head(3).T   # transpose: rows and columns swapped

,0,1,2
txn_id,T000001,T000002,T000003
ref_txn_id,<NA>,<NA>,<NA>
card_id,C1090,C1037,C1058
merchant,AMAZON MKTPLACE,WAITROSE 5418,SAINSBURYS S/MKT 7861
amount,132.66,41.65,30.22
currency,GBP,GBP,GBP
status,PENDING,PENDING,PENDING
txn_timestamp,2026-01-01 00:03:06,2026-01-01 00:25:35,01/01/2026 00:29
file_date,2026-01-01,2026-01-01,2026-01-01


In [148]:
# explode: a list in one cell becomes one row per item
pd.DataFrame({"card": ["C1", "C2"], "merchants": [["TESCO", "TFL"], ["NETFLIX"]]}).explode("merchants")

,card,merchants
0,C1,TESCO
0,C1,TFL
1,C2,NETFLIX


In [149]:
# get_dummies: one-hot encoding of a category
pd.get_dummies(tx["status"].head(4), prefix="s")

,s_PENDING
0,True
1,True
2,True
3,True


In [150]:
# wide_to_long reshapes columns named like x_2025, x_2026
pd.wide_to_long(pd.DataFrame({"id": [1, 2], "sales_2025": [10, 20], "sales_2026": [11, 21]}),
                stubnames="sales", i="id", j="year", sep="_")

,,sales
id,year,
1,2025,10
2,2025,20
1,2026,11
2,2026,21


**Use when:** a source is wide but you need long (or the reverse), for example a column per currency that must be joined on currency.

## 13. Combining tables
`merge` joins on keys like SQL. The danger is not the syntax but joins that silently change the row count. Always check rows before and after.

In [151]:
# Prepare small clean tables
cards = (cu.assign(snapshot_date=pd.to_datetime(cu["snapshot_date"]),
                   segment=cu["segment"].str.strip().str.title())
           .sort_values("snapshot_date").groupby("card_id").tail(1)[["card_id", "segment", "account_status"]])
tx_latest = (tx.drop_duplicates().sort_values(["txn_id", "file_date"]).groupby("txn_id").tail(1))
print(len(tx_latest), len(cards), cards["card_id"].is_unique)

15126 400 True


In [152]:
# left join: keep every transaction, add customer columns where found
m = tx_latest.merge(cards, on="card_id", how="left")
print(len(tx_latest), "->", len(m))
m["segment"].isna().sum()   # transactions on cards missing from the customer table

15126 -> 15126


np.int64(150)

In [153]:
# validate: pandas stops you if the key is not unique on the side you said it should be
tx_latest.merge(cards, on="card_id", how="left", validate="many_to_one").shape

(15126, 11)

In [154]:
# indicator: shows where each row came from (an anti-join and reconciliation tool)
chk = tx_latest.merge(cards, on="card_id", how="left", indicator=True)
chk["_merge"].value_counts()

,count
_merge,
both,14976
left_only,150
right_only,0


In [155]:
# Anti-join: rows in the left table with NO match in the right
chk[chk["_merge"] == "left_only"]["card_id"].nunique()

8

In [156]:
# how= options: inner (only matches), left, right, outer (everything)
for how in ["inner", "left", "outer"]:
    print(how, len(tx_latest.merge(cards, on="card_id", how=how)))

inner 14976
left 15126
outer 15126


In [157]:
# Different key names; keep both sides' columns apart with suffixes
a = pd.DataFrame({"k": [1, 2], "v": [10, 20]})
b = pd.DataFrame({"key": [1, 3], "v": [100, 300]})
a.merge(b, left_on="k", right_on="key", how="outer", suffixes=("_a", "_b"))

,k,v_a,key,v_b
0,1.0,10.0,1.0,100.0
1,2.0,20.0,NaN,NaN
2,NaN,NaN,3.0,300.0


Two things to notice. Text dates in two formats need parsing first (section 8), and GBP has no row in the rate table because it never needed one. We add it as 1.0 after the join.

In [158]:
# Join on several keys: the FX rate for the transaction DATE and CURRENCY
fx_long = (wide.ffill().rename_axis("rate_date").reset_index()
               .melt(id_vars="rate_date", var_name="currency", value_name="rate"))
iso = pd.to_datetime(tx_latest["txn_timestamp"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
uk = pd.to_datetime(tx_latest["txn_timestamp"], format="%d/%m/%Y %H:%M", errors="coerce")
# clean the currency code too, or "gbp" and "GBP " will not match the rate table
tx_fx = tx_latest.assign(rate_date=iso.fillna(uk).dt.normalize(),
                         currency=tx_latest["currency"].str.strip().str.upper())
j = tx_fx.merge(fx_long, on=["rate_date", "currency"], how="left", validate="many_to_one")
j["rate"] = j["rate"].where(j["currency"] != "GBP", 1.0)
print(len(tx_fx), "->", len(j), "| missing rates:", j["rate"].isna().sum())
j.assign(amount_gbp=j["amount"] * j["rate"])[["txn_id", "currency", "amount", "rate", "amount_gbp"]].head(3)

15126 -> 15126 | missing rates: 0


,txn_id,currency,amount,rate,amount_gbp
0,R00001,GBP,-36.73,1.0,-36.73
1,R00002,GBP,-20.60,1.0,-20.60
2,R00003,GBP,-20.66,1.0,-20.66


In [159]:
# concat: stack tables with the same columns (rows), or side by side (axis=1)
pd.concat([tx.head(2), tx.tail(2)], ignore_index=True)

,txn_id,ref_txn_id,card_id,merchant,amount,currency,status,txn_timestamp,file_date
0,T000001,<NA>,C1090,AMAZON MKTPLACE,132.66,GBP,PENDING,2026-01-01 00:03:06,2026-01-01
1,T000002,<NA>,C1037,WAITROSE 5418,41.65,GBP,PENDING,2026-01-01 00:25:35,2026-01-01
2,R00282,T014311,C1387,PRET A MANGER 8631,-4.00,GBP,REVERSED,2026-03-31 19:56:00,2026-03-31
3,R00283,T014370,C1300,SAINSBURYS S/MKT 5941,-45.33,GBP,REVERSED,2026-03-31 08:42:00,2026-03-31


In [160]:
# join: merge on the index
cards.set_index("card_id").join(tx_latest.groupby("card_id").size().rename("n_transactions")).head(3)

,segment,account_status,n_transactions
card_id,,,
C1265,Standard,ACTIVE,34
C1266,Standard,ACTIVE,40
C1267,Standard,ACTIVE,44


In [161]:
# merge_asof: match each row to the most recent earlier row (segment AT the time of purchase)
snaps = cu.assign(snapshot_date=pd.to_datetime(cu["snapshot_date"]),
                  segment=cu["segment"].str.strip().str.title())[["card_id", "snapshot_date", "segment"]].sort_values("snapshot_date")
left = tx_latest.assign(ts=pd.to_datetime(tx_latest["file_date"])).sort_values("ts")
pd.merge_asof(left, snaps, left_on="ts", right_on="snapshot_date", by="card_id", direction="backward")[["card_id", "ts", "snapshot_date", "segment"]].head(3)

,card_id,ts,snapshot_date,segment
0,C1218,2026-01-01,2025-12-29,Premium
1,C1131,2026-01-02,2025-12-29,Standard
2,C1198,2026-01-02,2025-12-29,Standard


In [162]:
# combine_first: fill gaps in one table from another
pd.Series([1, np.nan, 3]).combine_first(pd.Series([10, 20, 30]))

,0
0,1.0
1,20.0
2,3.0


**Use when:** enriching one table with another. Run `validate=` and compare row counts every time; most wrong totals in real pipelines come from a join that multiplied rows.

## 14. Validation and outliers
Cleaning is not finished until you have checked it. Write the checks as code so they run every time (this is what dbt tests do for you).

In [163]:
tx["amount"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

,amount
count,28829.000000
mean,31.980011
std,43.750476
min,-399.930000
1%,-1.692000
5%,3.024000
50%,20.580000
95%,95.620000
99%,213.112800
max,973.120000


In [164]:
# IQR rule: flag values far outside the middle 50%
q1, q3 = tx["amount"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
out = tx[(tx["amount"] < low) | (tx["amount"] > high)]
print(f"bounds {low:.1f} to {high:.1f}: {len(out)} rows flagged ({len(out)/len(tx):.1%})")

bounds -34.0 to 83.6: 1990 rows flagged (6.9%)


In [165]:
# z-score: distance from the mean in standard deviations
z = (tx["amount"] - tx["amount"].mean()) / tx["amount"].std()
(z.abs() > 4).sum()

np.int64(316)

In [166]:
# Outliers are questions, not errors. Look before you remove.
out.sort_values("amount", ascending=False).head(3)[["txn_id", "merchant", "amount", "status"]]

,txn_id,merchant,amount,status
2779,T001659,BOOKING.COM,973.12,PENDING
17758,T009301,BOOKING.COM,758.06,PENDING
18487,T009301,BOOKING.COM,758.06,POSTED


In [167]:
# assert: stop the run if a rule is broken
assert tx_latest["txn_id"].is_unique, "txn_id is not unique"
assert tx_latest["amount"].notna().all(), "missing amounts"
print("checks passed")

checks passed


In [168]:
# A reusable check that reports instead of stopping
def run_checks(checks):
    for name, ok in checks:
        print(("PASS  " if ok else "FAIL  ") + name)

run_checks([
    ("one row per txn_id",    tx_latest["txn_id"].is_unique),
    ("no missing card_id",    tx_latest["card_id"].notna().all()),
    ("statuses are known",    tx_latest["status"].isin(["PENDING", "POSTED", "REVERSED"]).all()),
    ("currencies are clean",  tx_latest["currency"].isin(["GBP", "EUR", "USD", "CHF"]).all()),
])

PASS  one row per txn_id
PASS  no missing card_id
PASS  statuses are known
FAIL  currencies are clean


The last check fails on purpose: we have not cleaned the currency codes (`gbp`, `GBP `) yet. That is what a check is for. Clean, then run it again.

In [169]:
tx_latest = tx_latest.assign(currency=tx_latest["currency"].str.strip().str.upper())
run_checks([("currencies are clean", tx_latest["currency"].isin(["GBP", "EUR", "USD", "CHF"]).all())])

PASS  currencies are clean


In [170]:
# Reconcile: rows in = rows out + rows removed on purpose
n_in = len(tx); n_dups = tx.duplicated().sum(); n_older = len(tx.drop_duplicates()) - len(tx_latest)
print(n_in, "=", len(tx_latest), "+", n_dups, "+", n_older, "->", n_in == len(tx_latest) + n_dups + n_older)

28829 = 15126 + 1269 + 12434 -> True


In [171]:
# Cross-check a total two ways
tx_latest.loc[tx_latest["status"] == "POSTED", "amount"].sum().round(2), \
tx_latest.query("status == 'POSTED'")["amount"].sum().round(2)

(np.float64(468621.78), np.float64(468621.78))

In [172]:
# Values that break a business rule: purchases should not be negative unless they are reversals
bad = tx_latest[(tx_latest["amount"] < 0) & (tx_latest["status"] != "REVERSED")]
len(bad)

0

In [173]:
# Referential check: every card in transactions exists in the customer table?
missing_cards = set(tx_latest["card_id"]) - set(cards["card_id"])
len(missing_cards), sorted(missing_cards)[:5]

(8, ['C9001', 'C9002', 'C9003', 'C9004', 'C9005'])

**Use when:** the last step of every cleaning stage, and again after every join. A check that never fails is still worth keeping; the one that fails saves you from a wrong number in a report.

## 15. Chaining steps into a pipeline
Instead of reassigning `df` over and over, write each step as a function and chain them with `.pipe()`. Each step becomes testable and the order is visible, which is how the `utilities.py` steps in the workshop are structured.

In [174]:
def drop_exact_duplicates(df):
    return df.drop_duplicates()

def clean_currency(df):
    return df.assign(currency=df["currency"].str.strip().str.upper())

def parse_timestamps(df):
    iso = pd.to_datetime(df["txn_timestamp"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
    uk = pd.to_datetime(df["txn_timestamp"], format="%d/%m/%Y %H:%M", errors="coerce")
    return df.assign(ts=iso.fillna(uk))

def keep_latest_status(df):
    rank = {"PENDING": 0, "POSTED": 1, "REVERSED": 2}
    return (df.assign(_r=df["status"].map(rank))
              .sort_values(["txn_id", "file_date", "_r"])
              .groupby("txn_id").tail(1)
              .drop(columns="_r"))

def add_flags(df):
    return df.assign(is_reversal=df["status"].eq("REVERSED"), zero_amount=df["amount"].eq(0))

silver_tx = (tx
             .pipe(drop_exact_duplicates)
             .pipe(clean_currency)
             .pipe(parse_timestamps)
             .pipe(keep_latest_status)
             .pipe(add_flags)
             .reset_index(drop=True))
silver_tx.shape

(15126, 12)

In [175]:
# Method chaining without functions: wrap in brackets, one step per line
(tx.drop_duplicates()
   .query("status == 'POSTED'")
   .assign(amount_k=lambda d: d["amount"] / 1000)
   .groupby("currency")["amount_k"].sum()
   .sort_values(ascending=False)
   .head(3))

,amount_k
currency,
GBP,347.66047
EUR,57.56220
USD,38.71991


In [176]:
# Log each step's row count so you can see where rows go
def logged(df, label):
    print(f"{label:<28}{len(df):>8,}")
    return df

(tx.pipe(logged, "raw")
   .drop_duplicates().pipe(logged, "after drop_duplicates")
   .pipe(keep_latest_status).pipe(logged, "after keep_latest_status")
   .query("status == 'POSTED'").pipe(logged, "posted only")
   .shape)

raw                           28,829
after drop_duplicates         27,560
after keep_latest_status      15,126
posted only                   14,388


(14388, 9)

**Use when:** a cleaning script gets longer than about ten lines. Named steps are easier to test, reorder and explain.

## 16. Saving results
Write the cleaned table somewhere durable. CSV is universal but loses types; Parquet keeps types, is smaller and faster to read.

In [177]:
silver_tx.to_csv("silver_transactions.csv", index=False)      # index=False: do not write row numbers
pd.read_csv("silver_transactions.csv").dtypes.head(6)         # note: dates come back as text

,0
txn_id,object
ref_txn_id,object
card_id,object
merchant,object
amount,float64
currency,object


In [178]:
pd.read_csv("silver_transactions.csv", parse_dates=["ts"]).dtypes["ts"]   # ask for dates back

dtype('<M8[ns]')

In [179]:
try:
    silver_tx.to_parquet("silver_transactions.parquet", index=False)
    back = pd.read_parquet("silver_transactions.parquet")
    print("parquet round trip keeps types:", back["ts"].dtype, "| size (KB):",
          os.path.getsize("silver_transactions.parquet") // 1024, "vs csv", os.path.getsize("silver_transactions.csv") // 1024)
except ImportError as e:
    print("Parquet needs pyarrow: !pip install pyarrow", e)

parquet round trip keeps types: datetime64[ns] | size (KB): 481 vs csv 1618


In [180]:
# Several sheets in one Excel file (needs openpyxl)
# with pd.ExcelWriter("out.xlsx") as w:
#     silver_tx.head(100).to_excel(w, sheet_name="transactions", index=False)

In [181]:
# Clean up the files this notebook made
for f in ["silver_transactions.csv", "silver_transactions.parquet"]:
    if os.path.exists(f):
        os.remove(f)

**Use when:** the end of a stage. Save with a clear name, record how many rows you wrote, and never overwrite the raw file.

## 17. Quick reference

In [182]:
ref = pd.DataFrame([
    ("Look at data",            "head, tail, sample, info, describe, dtypes, shape, nunique, value_counts"),
    ("Select columns / rows",   "df[cols], .loc, .iloc, boolean mask, query, isin, between, drop, rename"),
    ("Change types",            "astype, to_numeric, to_datetime, convert_dtypes, category"),
    ("Find missing",            "isna, notna, isna().sum(), isna().mean()"),
    ("Handle missing",          "fillna, ffill, bfill, interpolate, dropna, replace('', np.nan)"),
    ("Duplicates",              "duplicated, drop_duplicates(subset, keep), is_unique"),
    ("Clean text",              "str.strip, lower, upper, title, replace, contains, extract, split, len"),
    ("Recode values",           "map, replace, np.where, np.select, cut, qcut, where, clip"),
    ("New columns",             "df['x'] = ..., assign, apply (last resort), eval"),
    ("Dates",                   "to_datetime, .dt.year/month/day/dayofweek, to_period, normalize, date_range, resample"),
    ("Sort and rank",           "sort_values, nlargest, rank, idxmax, groupby().tail(1)"),
    ("Summarise",               "groupby + agg, size, transform, filter, pivot_table, crosstab"),
    ("Running calculations",    "shift, diff, pct_change, cumsum, cumcount, rolling, expanding"),
    ("Reshape",                 "pivot, melt, stack, unstack, explode, get_dummies, T"),
    ("Combine tables",          "merge (how, on, validate, indicator), concat, join, merge_asof, combine_first"),
    ("Check quality",           "assert, quantile, describe, isin, set difference, reconcile counts"),
    ("Build a pipeline",        "pipe, method chaining, one function per step"),
    ("Save",                    "to_csv, to_parquet, to_excel"),
], columns=["Task", "Functions"])
pd.set_option("display.max_colwidth", None, "display.width", 200)
ref.style.hide(axis="index").set_properties(**{"text-align": "left"})

Task,Functions
Look at data,"head, tail, sample, info, describe, dtypes, shape, nunique, value_counts"
Select columns / rows,"df[cols], .loc, .iloc, boolean mask, query, isin, between, drop, rename"
Change types,"astype, to_numeric, to_datetime, convert_dtypes, category"
Find missing,"isna, notna, isna().sum(), isna().mean()"
Handle missing,"fillna, ffill, bfill, interpolate, dropna, replace('', np.nan)"
Duplicates,"duplicated, drop_duplicates(subset, keep), is_unique"
Clean text,"str.strip, lower, upper, title, replace, contains, extract, split, len"
Recode values,"map, replace, np.where, np.select, cut, qcut, where, clip"
New columns,"df['x'] = ..., assign, apply (last resort), eval"
Dates,"to_datetime, .dt.year/month/day/dayofweek, to_period, normalize, date_range, resample"


### Habits worth keeping
- Look before you change: `info()`, `describe()`, `value_counts()`.
- Count rows before and after every step that can change the row count (drops, joins, groupbys).
- Convert types early; coerce, then count what failed.
- Prefer vectorised code to `apply` and loops.
- Write each rule once, as a named step, and say in words what it assumes.
- Keep the raw data untouched and make the cleaned table a new thing.